## Установка зависимостей/ядра
Без этого проект может не запуститься

1. `cd ~/old_home/querulus-main`
2. `python3 -m venv .venv`
3. `source .venv/bin/activate`
4. `python -m pip install -U pip setuptools wheel`
5. `pip install outboxml -e . ipykernel pandas numpy pyarrow scikit-learn catboost optuna matplotlib seaborn plotly kaleido environs pymssql openpyxl`
6. `python -m ipykernel install --user --name=querulus --display-name="Python (querulus)"`


# OutBoxML: модель 2 (TARGET_FREQ / TARGET_SEV)

Конфигурации моделей формируются из артефактов `train_loop_new` (отобранные признаки и параметры HPO).
Калибровка вероятностей не применяется: в производственный контур передаются сырые `predict_proba` / `predict`.

**Источник данных.** Приоритет — таблица Hive `models.querulus_df_final_3` (запись из `collect`). При недоступности Metastore или Spark используется локальный файл `data/processed/df_final_3.parquet`; источник фиксируется в логе (`[dataset] source=...`).

**Производственный ансамбль.** Повторное обучение на объединении исходного train и 85% более ранних наблюдений test (по дате). Финансовый эффект и порог сервиса оцениваются на оставшихся 15% наиболее поздних дат test (`prod_holdout`).

После prod-refit: графики FactorsPlot и cohort, рассылка `EMailDSResult`; для сервиса сохраняется parquet с колонками `preds_cf` / `preds_rg`.

На сервисе перед `prepare_dataset` применяются зафиксированные границы DQ из `querulus_dq_bounds_{version}.json` (`apply_frozen_dq_bounds`).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
OUTBOXML_ROOT = PROJECT_ROOT.parent.parent
for _p in (SRC, OUTBOXML_ROOT, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
print("PROJECT_ROOT", PROJECT_ROOT)
print("OUTBOXML_ROOT", OUTBOXML_ROOT)


In [ ]:
import json
import pickle
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format

from outboxml.core.prepared_datasets import PrepareDataset
from querulus.training.email_report import QuerulusEMailDSResult
from outboxml.data_subsets import DataPreprocessor
from outboxml.datasets_manager import DataSetsManager
from outboxml.export_results import ResultExport

from querulus.training.build_outboxml_configs import (
    dataframe_for_dsm,
    default_model_version,
    prepare_datasets_from_config,
    ensure_predictable_model,
    unwrap_estimator,
    write_outboxml_configs,
)
from querulus.training.outboxml_metrics import display_dsm_collect_metrics
from querulus.features.data_quality import write_service_dq_bounds
from querulus.fin_effect import (
    create_summary_table,
    export_business_html,
    print_best_threshold_report,
    resolve_fin_effect_config,
    run_fin_effect_pipeline,
)


In [ ]:
MODEL_VERSION = default_model_version(business="2", increment="v1")
HIVE_TABLE = "models.querulus_df_final_3"
# Путь локального parquet: кэш после успешного Hive и запасной источник при сбое
LOCAL_PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "df_final_3.parquet"
DATASET_PATH = LOCAL_PARQUET_PATH  # путь в OutBoxML JSON (после загрузки)
RESULTS_DIR = PROJECT_ROOT / "integration" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CF_NAME = f"querulus_cf_{MODEL_VERSION}"
RG_NAME = f"querulus_rg_{MODEL_VERSION}"
print("MODEL_VERSION", MODEL_VERSION)
print("HIVE_TABLE (приоритетный источник)", HIVE_TABLE)
print("LOCAL_PARQUET_PATH (кэш / fallback)", LOCAL_PARQUET_PATH)


In [ ]:
from querulus.dataset.hadoop import load_df_final

# Приоритет: Hive. При ошибке Metastore/Spark — LOCAL_PARQUET_PATH.
# Итог источника — блок «ИСТОЧНИК ДАТАСЕТА» в выводе load_df_final.
_df_raw, DATASET_SOURCE = load_df_final(
    hive_table=HIVE_TABLE,
    parquet_path=LOCAL_PARQUET_PATH,
    prefer_hive=True,
)
if DATASET_SOURCE.startswith("hive:"):
    LOCAL_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
    _df_raw.to_parquet(LOCAL_PARQUET_PATH, index=False)
    print(f"[dataset] кэш после Hive записан в parquet: {LOCAL_PARQUET_PATH}")
else:
    print(f"[dataset] Hive не использован; данные из файла: {LOCAL_PARQUET_PATH}")

DATASET_PATH = LOCAL_PARQUET_PATH
df = dataframe_for_dsm(_df_raw)
print("df.shape", df.shape, "| DATASET_SOURCE", DATASET_SOURCE)
built = write_outboxml_configs(
    df,
    version=MODEL_VERSION,
    parquet_path=str(DATASET_PATH.as_posix()),
)
periods = built["periods"]
print("built configs", built["cf_path"].name, built["rg_path"].name)
print("periods table")
display(periods["table"])
print("cutoff prod_holdout", periods["prod_cutoff"])


In [ ]:
def _patch_dsm_models(dsm):
    for name, res in dsm.get_result().items():
        res.model = ensure_predictable_model(res.model)


def _prepared_X(dsm, model_name, data, *, ignore_row_filter=False):
    """Обёртка: признаки DSM. Для severity на полном Test — ignore_row_filter=True."""
    from querulus.training.outboxml_metrics import prepare_dsm_features

    return prepare_dsm_features(
        dsm, model_name, data, ignore_row_filter=ignore_row_filter
    )


def _predict_cf(dsm, model_name, data):
    from querulus.training.outboxml_metrics import predict_dsm_series

    return predict_dsm_series(
        dsm,
        model_name,
        data,
        task_type="classification",
        ignore_row_filter=False,
    )


def _predict_rg(dsm, model_name, data):
    """Severity на всех строках data (без фильтра TARGET_SEV > 0 из обучения)."""
    from querulus.training.outboxml_metrics import predict_dsm_series

    return predict_dsm_series(
        dsm,
        model_name,
        data,
        task_type="regression",
        ignore_row_filter=True,
    )


def _fin_effect_table(df_all, index, proba, sev, *, threshold=None, title=""):
    """Финэффект: proba и sev на одном index (полный holdout), сводка = агрегация frame."""
    cfg = resolve_fin_effect_config(
        df_all,
        frequency_target="TARGET_FREQ",
        severity_target="TARGET_SEV",
    )
    common = (
        pd.Index(index)
        .intersection(proba.dropna().index)
        .intersection(sev.dropna().index)
        .intersection(df_all.index)
    )
    if len(common) == 0:
        raise ValueError(
            "Нет пересечения index с proba/sev: проверьте index предсказаний."
        )
    coverage = len(common) / max(len(pd.Index(index)), 1)
    if coverage < 0.95:
        raise ValueError(
            f"pred покрывает только {len(common)}/{len(index)} строк ({coverage:.1%}). "
            "Для severity нужен predict без data_filter_condition "
            "(ignore_row_filter=True в _predict_rg)."
        )
    if len(common) < len(index):
        print(
            f"[fin_effect] строк с pred: {len(common)}/{len(index)} "
            f"(отброшено без proba/sev: {len(index) - len(common)})"
        )
    aligned = df_all.loc[common]
    fe = run_fin_effect_pipeline(
        aligned,
        proba.reindex(common),
        sev.reindex(common),
        aligned["TARGET_FREQ"],
        threshold=threshold,
        config=cfg,
    )
    if title:
        display(Markdown(f"### {title}"))
    print_best_threshold_report(fe)
    summary = fe.summary_table(cfg)
    display(summary.style.format("{:,.0f}", subset=summary.columns[3:], na_rep="—"))
    print(
        f"проверка Σ: model={summary['ФИН. ЭФФЕКТ МОДЕЛЬ'].sum():,.0f} "
        f"(отчёт {fe.model_effect_total:,.0f}), "
        f"fact={summary['ФИН. ЭФФЕКТ ФАКТ'].sum():,.0f} "
        f"(отчёт {fe.fact_effect_total:,.0f}), "
        f"экон.={summary['Экономия'].sum():,.0f} "
        f"(отчёт net {fe.net_effect:,.0f})"
    )
    n_pos = int((aligned["TARGET_FREQ"] == 1).sum())
    n_neg = int((aligned["TARGET_FREQ"] == 0).sum())
    print(f"holdout в расчёте: n={len(aligned)} pos={n_pos} neg={n_neg}")
    return fe, summary


## Parity: DSM на train_core ∪ val; test = полный holdout

Модели parity не входят в производственный ансамбль. Они используются для сопоставления финансового эффекта с блоком C3 ноутбука `collect` на том же полном holdout Test.


In [ ]:
from configs import config as querulus_outboxml_config

dsm_cf = DataSetsManager(
    config_name=str(built["cf_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["cf_path"]),
)
dsm_cf.load_dataset(data=df)
dsm_cf.fit_models()
_patch_dsm_models(dsm_cf)

dsm_rg = DataSetsManager(
    config_name=str(built["rg_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["rg_path"]),
)
dsm_rg.load_dataset(data=df)
dsm_rg.fit_models()
_patch_dsm_models(dsm_rg)
display_dsm_collect_metrics(dsm_cf, CF_NAME, task_type="classification", title=f"parity {CF_NAME}")
display_dsm_collect_metrics(dsm_rg, RG_NAME, task_type="regression", title=f"parity {RG_NAME}")
print("parity fit done", CF_NAME, RG_NAME)


### Parity vs prod: почему метрики «prod» хуже при прямом сравнении

Таблицы `parity …` и `prod …` выше **не сопоставимы напрямую**:

1. **Разный test.** Parity DSM считает test на **всем** holdout Test. Prod DSM — только на **prod_holdout** (≈15% самых свежих дат test, после `prod_cutoff`). Свежий хвост обычно сложнее → выше MAE/RMSE, ниже R².
2. **Разный train.** Parity учится на train∪val. Prod refit — на train + 85% test **до cutoff** (те же строки, которые parity видит только на test). Train-метрики prod тоже на другой выборке.
3. **Одинаковый gini на test** у parity/prod в отдельных таблицах — совпадение на разных срезах; для честного сравнения смотри таблицу ниже.

Ниже — **одна parity-модель**, train = parity train, test = полный parity test **и** prod holdout в одинаковых условиях (один estimator, один фильтр `TARGET_SEV > 0`).

In [ ]:
from querulus.training.outboxml_metrics import display_dsm_collect_metrics_cross_test

_parity_test_idx = periods["splits"].test
_prod_holdout_idx = periods["prod_holdout_idx"]

display_dsm_collect_metrics_cross_test(
    dsm_rg,
    RG_NAME,
    df,
    task_type="regression",
    test_slices={
        "parity_test": _parity_test_idx,
        "prod_test": _prod_holdout_idx,
    },
    title=f"parity model {RG_NAME}: train / parity_test / prod_holdout",
    ignore_row_filter=False,
)

## Финансовый эффект: parity (полный Test)

Предсказания frequency и severity строятся на полном holdout Test. Порог классификации подбирается внутри расчёта финансового эффекта по критерию `net_effect`.

Если в окружении ядра уже есть результат `fin_effect_b` из `collect`, он выводится для сопоставления с parity.


In [ ]:
_fe_collect = globals().get("fin_effect_b")
if _fe_collect is not None:
    display(Markdown("### Collect HPO / блок C3 (полный Test)"))
    print_best_threshold_report(_fe_collect)
    _cfg_c = globals().get("FIN_EFFECT_CONFIG_B")
    if _cfg_c is not None:
        _sum_c = create_summary_table(_fe_collect.frame, _cfg_c)
        display(_sum_c.style.format("{:,.0f}", subset=_sum_c.columns[3:], na_rep="—"))

# Предсказания для финансового эффекта на полном holdout Test
test_idx = periods["splits"].test
proba_test = _predict_cf(dsm_cf, CF_NAME, df.loc[test_idx])
sev_test = _predict_rg(dsm_rg, RG_NAME, df.loc[test_idx])

fe_parity, _ = _fin_effect_table(
    df,
    test_idx,
    proba_test,
    sev_test,
    title="OutBoxML parity: финансовый эффект на полном Test (сырые proba/sev)",
)

_fe_html = _fe_collect if _fe_collect is not None else fe_parity
_cfg_html = globals().get("FIN_EFFECT_CONFIG_B")
if _cfg_html is None:
    _cfg_html = resolve_fin_effect_config(
        df, frequency_target="TARGET_FREQ", severity_target="TARGET_SEV"
    )
_html_path = export_business_html(
    _fe_html,
    _cfg_html,
    path=PROJECT_ROOT / "notebooks" / "fin_effect_detailed.html",
    subtitle=(
        "Collect: финансовый эффект на полном Test"
        if _fe_collect is not None
        else "OutBoxML parity: сырые предсказания, полный Test"
    ),
)
print(f"HTML для бизнеса: {_html_path}")


## Prod-refit (модели производственного ансамбля)

Обучение: исходный train и наблюдения test с датой не позже cutoff (85% более ранних строк test по `PAYMENT_ORDER_DATE_TIME`).

В этом разделе для prod-моделей выводятся:

1. метрики DSM в формате collect (для классификации — порог 0.5; это не порог сервиса);
2. финансовый эффект и подобранный порог сервиса на `prod_holdout` (15% наиболее поздних дат test).


In [ ]:
dsm_cf_prod = DataSetsManager(
    config_name=str(built["cf_prod_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["cf_prod_path"]),
)
dsm_cf_prod.load_dataset(data=df)
dsm_cf_prod.fit_models()
_patch_dsm_models(dsm_cf_prod)

dsm_rg_prod = DataSetsManager(
    config_name=str(built["rg_prod_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["rg_prod_path"]),
)
dsm_rg_prod.load_dataset(data=df)
dsm_rg_prod.fit_models()
_patch_dsm_models(dsm_rg_prod)

display_dsm_collect_metrics(
    dsm_cf_prod, CF_NAME, task_type="classification", title=f"prod {CF_NAME}"
)
display_dsm_collect_metrics(
    dsm_rg_prod, RG_NAME, task_type="regression", title=f"prod {RG_NAME}"
)

prod_holdout_idx = periods["prod_holdout_idx"]

proba_prod = _predict_cf(dsm_cf_prod, CF_NAME, df.loc[prod_holdout_idx])
sev_prod = _predict_rg(dsm_rg_prod, RG_NAME, df.loc[prod_holdout_idx])
fe_prod, _ = _fin_effect_table(
    df,
    prod_holdout_idx,
    proba_prod,
    sev_prod,
    title="Prod-refit: финансовый эффект на prod_holdout (сырые proba/sev)",
)
print(
    "Порог сервиса (best_threshold из финансового эффекта на prod_holdout):",
    float(fe_prod.best_threshold),
)


## FactorsPlot, cohort и QuerulusEMailDSResult (prod)


In [ ]:
def _plot_features(dsm, model_name, *, n_num=6, n_cat=6):
    """Признаки для FactorsPlot из data_subset (в JSON targetslices пустые)."""
    subset = dsm.get_result()[model_name].data_subset
    nums = list(subset.features_numerical or [])[:n_num]
    cats = list(subset.features_categorical or [])[:n_cat]
    return nums + cats


def _show_figure(fig, title: str):
    if fig is None:
        print(f"[warn] {title}: figure is None")
        return
    show = getattr(fig, "show", None)
    if callable(show):
        show()
    else:
        display(fig)
    print(title, type(fig))


def _show_factors(export, model_name, features, *, bins=5):
    """По одной фиче: OutBoxML возвращает только последний figure из списка."""
    for feat in features:
        fig = export.plots(
            model_name=model_name,
            features=[feat],
            plot_type=1,
            bins_for_numerical_features=bins,
            use_exposure=False,
            only_test=True,
        )
        _show_figure(fig, f"FactorsPlot {model_name}: {feat}")


export_cf = ResultExport(ds_manager=dsm_cf_prod, config=querulus_outboxml_config)
export_rg = ResultExport(ds_manager=dsm_rg_prod, config=querulus_outboxml_config)
cf_plot_feats = _plot_features(dsm_cf_prod, CF_NAME)
rg_plot_feats = _plot_features(dsm_rg_prod, RG_NAME)
print("FactorsPlot features CF:", cf_plot_feats)
print("FactorsPlot features RG:", rg_plot_feats)

_show_factors(export_cf, CF_NAME, cf_plot_feats)
_show_factors(export_rg, RG_NAME, rg_plot_feats)

fig_cf_cohort = export_cf.plots(
    model_name=CF_NAME,
    plot_type=2,
    use_exposure=False,
    only_test=True,
    cut_min_value=0.1,
    cut_max_value=0.9,
    samples=100,
    cohort_base="model",
)
fig_rg_cohort = export_rg.plots(
    model_name=RG_NAME,
    plot_type=2,
    use_exposure=False,
    only_test=True,
    cut_min_value=0.1,
    cut_max_value=0.9,
    samples=100,
    cohort_base="model",
)
_show_figure(fig_cf_cohort, "Cohort plot_type=2 CF")
_show_figure(fig_rg_cohort, "Cohort plot_type=2 RG")

_prod_results = {}
_prod_results.update(dsm_cf_prod.get_result())
_prod_results.update(dsm_rg_prod.get_result())
try:
    QuerulusEMailDSResult(
        config=querulus_outboxml_config,
        ds_manager_result=_prod_results,
    ).success_mail(group_name=f"querulus_{MODEL_VERSION}")
    print("QuerulusEMailDSResult: письмо отправлено")
except Exception as exc:
    print(f"[warn] QuerulusEMailDSResult не отправлено: {type(exc).__name__}: {exc}")


## Экспорт артефактов для сервиса (prod)

Запись pickle frequency/severity и ансамбля, файла границ DQ, parquet `df_for_service` (исходный датасет с колонками `preds_cf` / `preds_rg`) и JSON с метаданными (`querulus_meta_*.json`), включая `best_threshold` из раздела Prod-refit.


In [ ]:
cf_export = dsm_cf_prod.get_result()[CF_NAME].dict_for_prod_export()
rg_export = dsm_rg_prod.get_result()[RG_NAME].dict_for_prod_export()
cf_export["model"] = ensure_predictable_model(cf_export["model"])
rg_export["model"] = ensure_predictable_model(rg_export["model"])

cf_pkl = RESULTS_DIR / f"querulus_cf_for_prod_{MODEL_VERSION}.pickle"
rg_pkl = RESULTS_DIR / f"querulus_rg_for_prod_{MODEL_VERSION}.pickle"
ans_pkl = RESULTS_DIR / f"querulus_ansamble_{MODEL_VERSION}.pickle"
dq_report = PROJECT_ROOT / "data" / "processed" / "data_quality_report.json"
dq_bounds_pkl = RESULTS_DIR / f"querulus_dq_bounds_{MODEL_VERSION}.json"
if dq_report.exists():
    write_service_dq_bounds(
        dq_bounds_pkl,
        model_version=MODEL_VERSION,
        report_path=dq_report,
    )
    print("dq_bounds", dq_bounds_pkl)
else:
    print("[warn] data_quality_report.json не найден — querulus_dq_bounds не записан")
    dq_bounds_pkl = None

cf_pkl.write_bytes(pickle.dumps([cf_export]))
rg_pkl.write_bytes(pickle.dumps([rg_export]))
ensemble = [deepcopy(cf_export), deepcopy(rg_export)]
ans_pkl.write_bytes(pickle.dumps(ensemble))

df_service = df.copy()
df_service["preds_cf"] = _predict_cf(dsm_cf_prod, CF_NAME, df)
df_service["preds_rg"] = _predict_rg(dsm_rg_prod, RG_NAME, df)
service_df_path = PROJECT_ROOT / "data" / "processed" / f"df_for_service_{MODEL_VERSION}.parquet"
df_service.to_parquet(service_df_path, index=True)
print(
    "df_for_service",
    service_df_path,
    "shape",
    df_service.shape,
    "preds_cf na%",
    float(df_service["preds_cf"].isna().mean()),
    "preds_rg na%",
    float(df_service["preds_rg"].isna().mean()),
)

meta = {
    "model_version": MODEL_VERSION,
    "periods": {
        k: list(v) if isinstance(v, tuple) else v
        for k, v in periods.items()
        if k in {
            "parity_train_period",
            "parity_test_period",
            "prod_train_period",
            "prod_test_period",
            "prod_cutoff",
            "date_column",
            "cal_period",
        }
    },
    "cf_name": CF_NAME,
    "rg_name": RG_NAME,
    "best_threshold": float(fe_prod.best_threshold),
    "calibration": None,
    "artifacts": {
        "cf": str(cf_pkl),
        "rg": str(rg_pkl),
        "ensemble": str(ans_pkl),
        "dq_bounds": str(dq_bounds_pkl) if dq_bounds_pkl else None,
        "df_for_service": str(service_df_path),
    },
    "preds_cf_col": "preds_cf",
    "preds_rg_col": "preds_rg",
}
(RESULTS_DIR / f"querulus_meta_{MODEL_VERSION}.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print(
    "wrote",
    cf_pkl.name,
    rg_pkl.name,
    ans_pkl.name,
    dq_bounds_pkl.name if dq_bounds_pkl else None,
    service_df_path.name,
)
print("best_threshold=", meta["best_threshold"])
